# 01 · Dataset — FFHQ pipeline
Verify acquisition, decoding, normalization, mirroring and throughput before any GAN code runs.

In [ ]:
import os, sys
# ---- platform auto-detect: the same notebook runs on Colab and Kaggle ----
PLATFORM = "kaggle" if os.path.exists("/kaggle/input") else "colab"
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if (PLATFORM == "colab" or PLATFORM == "kaggle") and not os.path.exists("src"):
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/Ravikishore710/styleforge3-T.git"], check=True)
    os.chdir("styleforge3-T")
sys.path.insert(0, os.path.abspath("."))
!pip install -q -r requirements.txt
import tensorflow as tf
print("platform:", PLATFORM, "| TF:", tf.__version__,
      "| GPU:", tf.config.list_physical_devices("GPU"))
# Kaggle: enable GPU (Settings -> Accelerator -> GPU P100) and add the FFHQ
# dataset to /kaggle/input, or run scripts/prepare_ffhq.py --source folder.

## Prepare data
Full FFHQ: `python scripts/prepare_ffhq.py --source drive --resolution 256` (or copy from a Kaggle dataset / local folder).

In [ ]:
from src.config import load_config
from src.data.ffhq import build_dataset, write_manifest
cfg = load_config('configs/ffhq_256.yaml')
ds, num = build_dataset(cfg)
print('images:', num, '| resolution:', cfg['img_resolution'])
print(write_manifest(cfg, 'data/manifest.json'))

In [ ]:
import matplotlib.pyplot as plt
batch = next(iter(ds)).numpy()
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for ax, img in zip(axes.flat, batch[:16]):
    ax.imshow((img + 1) / 2); ax.axis('off')
plt.tight_layout(); plt.show()
print('range:', batch.min(), batch.max(), '| shape:', batch.shape)

In [ ]:
import time
it = iter(ds); t0 = time.time()
for _ in range(20): _ = next(it)
print(f'throughput: {20 * cfg["batch_size"] / (time.time() - t0):.0f} img/s')